# Chapter 14 — Backtest Statistics

AFML Chapter 14 is mostly a catalogue of the statistics used to report and judge a
backtest. Four sections have actual code snippets (14.1 bet timing, 14.2 holding
period, 14.3 HHI concentration, 14.4 drawdown/time-under-water); PSR and DSR (14.7.2
and 14.7.3) have formulas but no book snippet; classification scores (14.8) are
standard metrics, reinterpreted for meta-labeling.

**Scope note (recorded in project memory, 2026-07-20):** Implementation shortfall
(14.6) and attribution (14.9) are deliberately out of scope for now — both are
prose-only sections with no code, and both need data this pipeline doesn't produce
(real broker fees/fill prices for 14.6; a multi-asset portfolio with disjoint risk
buckets for 14.9). Revisit once that data exists.

This notebook runs everything against **real** data: Ch12's actual CPCV output on
the enriched real BTC/TUSD table (Ch03–05 + Ch19's microstructural features), not
synthetic placeholders.

In [ ]:
import os
import sys

# --- EDIT THIS LINE for your machine ---
AFML_ROOT = r'C:\ws\AFML'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(AFML_ROOT, 'ch14', 'backtest_statistics'))
from backtest_statistics import (
    getBetTiming, getHoldingPeriod, hhi_concentration_stats, computeDD_TuW,
    probabilistic_sharpe_ratio, expected_max_sharpe, deflated_sharpe_ratio,
)
from classification_scores import classification_scores

sys.path.insert(0, os.path.join(AFML_ROOT, 'ch12', 'cpcv'))
from chapter_12_cpcv import (
    load_data, run_cpcv, path_to_signal_and_returns,
    N_GROUPS, K_TEST_GROUPS, PCT_EMBARGO, SVC_C, SVC_GAMMA, RANDOM_STATE,
)

INPUT_DIR = os.path.join(AFML_ROOT, 'input_data')
DSR_SIGNIFICANCE = 0.95

## Shared setup: re-run Ch12's real CPCV

Same call, same `random_state` Ch12 already uses correctly (confirmed deterministic
across two consecutive real-machine runs, July 20 session) — so this reproduces
Ch12's real 5-path output byte-for-byte, giving Ch14 a genuine real position/return
series and 5 genuine real Sharpe-ratio "trials" to work with.

In [ ]:
X, y, w, t1, ret = load_data()
path_prob, path_pred, group_bounds, phi = run_cpcv(
    X, y, w, t1, n_groups=N_GROUPS, k=K_TEST_GROUPS, pct_embargo=PCT_EMBARGO,
    C=SVC_C, gamma=SVC_GAMMA, random_state=RANDOM_STATE,
)
path_data = {}
path_sharpes = []
for p in range(1, phi + 1):
    signal, pos_returns, sharpe = path_to_signal_and_returns(t1, ret, path_prob[p], path_pred[p])
    path_data[p] = (signal, pos_returns, sharpe)
    path_sharpes.append(sharpe)

print(f'{phi} real CPCV paths loaded. Sharpes: {[round(s, 4) for s in path_sharpes]}')

## Part A — Bet timing, holding period, HHI, drawdown/TuW (real CPCV path 1)

**Plain English:** A "bet" isn't the same as a trade — if you're long for five days
straight, that's one bet held over five days, not five separate trades. Snippet 14.1
finds the timestamps where a bet *ends* (position goes flat, or flips sign). Snippet
14.2 uses those timestamps (plus a running weighted-entry-time trick) to estimate how
long, on average, a bet is held.

HHI (Herfindahl-Hirschman Index, snippet 14.3) measures concentration: if one bet
accounts for almost all of your positive PnL, HHI on positive returns is close to 1
(risky — you're one lucky trade away from your whole track record). If PnL is spread
evenly across many bets, HHI is close to 0.

Drawdown (DD) and time-under-water (TuW), snippet 14.4, ask: from any given peak in
your cumulative PnL (high-water mark), how far down did you fall before recovering,
and how long did that take?

**Math:** HHI is built from return weights $w_t = r_t / \sum_t r_t$:

$$h = \frac{\sum_t w_t^2 - \|w\|^{-1}}{1 - \|w\|^{-1}}$$

DD is $1 - \min(\text{pnl})/\text{hwm}$ between consecutive high-water marks; TuW is
the elapsed time between one high-water mark and the next.

In [ ]:
signal1, pos_returns1, sharpe1 = path_data[1]

bets = getBetTiming(signal1)
print(f'{len(bets)} distinct bets identified from path 1\'s real position series '
      f'({(signal1 != 0).sum()}/{len(signal1)} nonzero positions)')

hp = getHoldingPeriod(signal1)
print(f'Average holding period: {hp:.4f} days')

bet_ret = pos_returns1[pos_returns1 != 0]
hhi = hhi_concentration_stats(bet_ret)
print(f'HHI (positive-return concentration): {hhi["hhi_positive"]:.4f}')
print(f'HHI (negative-return concentration): {hhi["hhi_negative"]:.4f}')
print(f'HHI (bets-per-month concentration): {hhi["hhi_time"]} '
      f'(NaN expected -- real data spans <2 full calendar months)')

cum_pnl = pos_returns1.cumsum()
dd, tuw = computeDD_TuW(cum_pnl, dollars=True)
print(f'\n{len(dd)} drawdown episode(s) identified')
if len(dd) > 0:
    print(f'95th-percentile drawdown: {dd.quantile(0.95):.4f}')
if len(tuw) > 0:
    print(f'95th-percentile time-under-water: {tuw.quantile(0.95) * 365.25:.2f} days')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cum_pnl.index, cum_pnl.values, color='steelblue', label='cumulative PnL')
ax.plot(cum_pnl.index, cum_pnl.expanding().max().values, color='firebrick',
        linestyle='--', alpha=0.6, label='high-water mark')
ax.set_title('Path 1: cumulative PnL and high-water mark (real BTC/TUSD)')
ax.legend()
fig.autofmt_xdate()
plt.show()

## Part B — PSR / DSR across all 5 real CPCV paths

**Plain English:** A Sharpe ratio from one short, possibly-skewed backtest can look
good by luck. PSR asks: "given how few observations I have, and how skewed/fat-tailed
they are, how confident can I really be that my Sharpe beats some benchmark?" DSR goes
further: "I didn't run *one* backtest, I ran several (5 real CPCV paths) — correct for
that, the same way a p-value needs correcting when you've run multiple tests."

**Math:**

$$\text{PSR}[SR^*] = Z\!\left[\frac{(\hat{SR} - SR^*)\sqrt{T-1}}{\sqrt{1 - \hat\gamma_3 \hat{SR} + \frac{\hat\gamma_4 - 1}{4}\hat{SR}^2}}\right]$$

$$SR^* = \sqrt{V[\{\hat{SR}_n\}]}\left[(1-\gamma)Z^{-1}\!\left[1-\tfrac{1}{N}\right] + \gamma Z^{-1}\!\left[1-\tfrac{1}{Ne}\right]\right]$$

Here, $V[\{\hat{SR}_n\}]$ is the real cross-path Sharpe variance from Ch12's 5 genuine
backtest paths — not a synthetic placeholder. $N=5$ is a real trial count.

In [ ]:
T = len(bet_ret)
skew = bet_ret.skew()
kurtosis = bet_ret.kurtosis() + 3  # pandas kurtosis() is EXCESS kurtosis; book's gamma4 is raw (=3 for Gaussian)
var_sr_trials = np.var(path_sharpes, ddof=1)
N = len(path_sharpes)
sr_star = expected_max_sharpe(var_sr_trials, N)

print(f'Real cross-path Sharpe variance: {var_sr_trials:.6f} (N={N} trials)')
print(f'Expected max Sharpe under H0 (SR*): {sr_star:.4f}')
print(f'(skew={skew:.4f}, kurtosis={kurtosis:.4f}, T={T} nonzero bets, from path 1)\n')

dsr_results = []
for p in range(1, phi + 1):
    _, _, sharpe = path_data[p]
    psr0 = probabilistic_sharpe_ratio(sharpe, 0., T, skew, kurtosis)
    dsr = deflated_sharpe_ratio(sharpe, var_sr_trials, N, T, skew, kurtosis)
    survives = dsr > DSR_SIGNIFICANCE
    dsr_results.append({'path': p, 'sharpe': sharpe, 'psr_vs_zero': psr0, 'dsr': dsr, 'survives_dsr': survives})
    print(f'path {p}: Sharpe={sharpe:+.4f}  PSR[0]={psr0:.4f}  DSR={dsr:.4f}  '
          f'{"SURVIVES (>0.95)" if survives else "does not survive deflation"}')

n_survive = sum(r['survives_dsr'] for r in dsr_results)
print(f'\n{n_survive}/5 paths survive DSR at the {DSR_SIGNIFICANCE} significance level.')

fig, ax = plt.subplots(figsize=(7, 4))
paths = [r['path'] for r in dsr_results]
dsrs = [r['dsr'] for r in dsr_results]
ax.bar(paths, dsrs, color='steelblue')
ax.axhline(DSR_SIGNIFICANCE, color='firebrick', linestyle='--', label=f'{DSR_SIGNIFICANCE} significance')
ax.set_title('DSR per real CPCV path')
ax.set_xlabel('path')
ax.set_ylabel('DSR')
ax.set_ylim(0, 1)
ax.legend()
plt.show()

**Real-data finding:** even paths whose PSR[0] alone looked reasonably strong (e.g.
path 2 at 0.92, close to the 0.95 significance bar against a naive zero benchmark)
fail once DSR corrects for having genuinely run N=5 trials. This is a fourth,
differently-mechanised line of evidence pointing the same direction as Ch11's PBO
(~0.73), Ch12's own CPCV Sharpe spread (mean 0.067, straddling zero), and Ch13's
O-U non-stationarity finding: **this real feature set/model combination shows no
backtest result that survives rigorous statistical scrutiny.** Not a code bug —
a real, book-consistent result (this is exactly the kind of overfitting DSR exists
to catch — Snippet 14.5, the third law of backtesting).

## Part C — Classification scores (14.8)

**Plain English:** In meta-labeling, the primary model finds opportunities; a
secondary classifier decides whether to take them. Plain accuracy can mislead under
class imbalance — a classifier that always predicts "no" can still score high
accuracy if "no" is the common case. F1 and negative log-loss correct for that.
Table 14.1's degenerate cases (all-1s observed/predicted, etc.) show precision or
recall can be genuinely *undefined* (NaN), not just zero — implemented directly from
the book's confusion-matrix formulas rather than relying on sklearn defaults, which
don't consistently distinguish "0" from "undefined" across the sklearn versions this
project has used.

In [ ]:
y_binary = (y.values > 0).astype(int)  # pipeline uses {-1,+1} -> remap to {0,1}
pred1 = path_pred[1]
pred1_binary = (pred1 > 0).astype(int)
prob1 = path_prob[1]
proba2col = np.zeros((len(prob1), 2))
for i in range(len(prob1)):
    if pred1_binary[i] == 1:
        proba2col[i] = [1 - prob1[i], prob1[i]]
    else:
        proba2col[i] = [prob1[i], 1 - prob1[i]]

scores = classification_scores(y_binary, pred1_binary, y_proba=proba2col)
for k, v in scores.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')

Accuracy well under 0.5 on real out-of-sample predictions is itself informative —
consistent with the pipeline's recurring finding (Ch08/09's thin CV scores, Ch11's
PBO, Ch12's Sharpe spread, Ch13's non-stationarity, and now Ch14's DSR result) that
this real feature set/model combination carries little genuine predictive signal on
this real BTC/TUSD data.

## TDD results

Real machine (`mlfinlab` env), 2026-07-21:

```
(mlfinlab) PS C:\ws\AFML\ch14\backtest_statistics> python -m pytest -v
platform win32 -- Python 3.10.20, pytest-9.0.3, pluggy-1.6.0
rootdir: C:\ws\AFML\ch14\backtest_statistics
collected 31 items

test_backtest_statistics.py::TestGetBetTiming::test_flattening_and_flip PASSED                                    [  3%]
test_backtest_statistics.py::TestGetBetTiming::test_last_bet_appended_when_no_natural_end PASSED                  [  6%]
test_backtest_statistics.py::TestGetBetTiming::test_no_double_count_when_last_index_already_a_bet PASSED           [  9%]
test_backtest_statistics.py::TestGetHoldingPeriod::test_single_trade_known_duration PASSED                        [ 12%]
test_backtest_statistics.py::TestGetHoldingPeriod::test_two_trades_weighted_average PASSED                        [ 16%]
test_backtest_statistics.py::TestGetHoldingPeriod::test_never_enters_position_returns_nan PASSED                  [ 19%]
test_backtest_statistics.py::TestGetHHI::test_uniform_returns_near_zero PASSED                                    [ 22%]
test_backtest_statistics.py::TestGetHHI::test_single_dominant_return_near_one PASSED                              [ 25%]
test_backtest_statistics.py::TestGetHHI::test_small_sample_returns_nan PASSED                                     [ 29%]
test_backtest_statistics.py::TestGetHHI::test_bounded_zero_to_one PASSED                                          [ 32%]
test_backtest_statistics.py::TestHHIConcentrationStats::test_splits_positive_and_negative_correctly PASSED        [ 35%]
test_backtest_statistics.py::TestComputeDDTuW::test_single_drawdown_known_value PASSED                            [ 38%]
test_backtest_statistics.py::TestComputeDDTuW::test_two_drawdowns_known_values_and_tuw PASSED                     [ 41%]
test_backtest_statistics.py::TestComputeDDTuW::test_dollars_mode_matches_pct_mode_relationship PASSED             [ 45%]
test_backtest_statistics.py::TestComputeDDTuW::test_monotonically_increasing_series_has_no_drawdown PASSED        [ 48%]
test_backtest_statistics.py::TestProbabilisticSharpeRatio::test_sr_hat_equals_benchmark_is_one_half PASSED        [ 51%]
test_backtest_statistics.py::TestProbabilisticSharpeRatio::test_matches_manual_formula PASSED                     [ 54%]
test_backtest_statistics.py::TestProbabilisticSharpeRatio::test_increases_with_sr_hat PASSED                      [ 58%]
test_backtest_statistics.py::TestProbabilisticSharpeRatio::test_increases_with_longer_track_record PASSED         [ 61%]
test_backtest_statistics.py::TestProbabilisticSharpeRatio::test_decreases_with_fatter_tails PASSED                [ 64%]
test_backtest_statistics.py::TestExpectedMaxSharpe::test_increases_with_n_trials PASSED                           [ 67%]
test_backtest_statistics.py::TestExpectedMaxSharpe::test_increases_with_trial_variance PASSED                     [ 70%]
test_backtest_statistics.py::TestExpectedMaxSharpe::test_matches_manual_formula PASSED                            [ 74%]
test_backtest_statistics.py::TestDeflatedSharpeRatio::test_matches_psr_composed_with_expected_max_sharpe PASSED   [ 77%]
test_backtest_statistics.py::TestDeflatedSharpeRatio::test_approaches_psr_zero_as_trial_variance_shrinks PASSED   [ 80%]
test_backtest_statistics.py::TestDeflatedSharpeRatio::test_more_trials_deflates_more PASSED                       [ 83%]
test_classification_scores.py::TestClassificationScores::test_observed_all_ones PASSED                            [ 87%]
test_classification_scores.py::TestClassificationScores::test_observed_all_zeros PASSED                           [ 90%]
test_classification_scores.py::TestClassificationScores::test_predicted_all_ones PASSED                           [ 93%]
test_classification_scores.py::TestClassificationScores::test_predicted_all_zeros PASSED                          [ 96%]
test_classification_scores.py::TestClassificationScores::test_neg_log_loss_included_when_proba_given PASSED       [100%]

============================== 31 passed in 3.98s ==============================
```

**Note on pandas versions:** the 5 tests in `TestHHIConcentrationStats` /
`TestComputeDDTuW` fail under pandas 3.x, which removed `'M'` as an offset alias
and `'Y'` as a `timedelta64` unit. Both are valid under this project's pandas
1.5.3 and match the book's own printed snippets — confirmed passing above on the
real machine.